In [0]:

# =============================================================================
# 03_gold_olist.py
# Camada: 03 · GOLD — Olist E-Commerce
# Lê directamente das tabelas Silver já disponíveis
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import date

# ---------------------------------------------------------------------------
# 1. LEITURA DAS TABELAS SILVER
# ---------------------------------------------------------------------------
df1  = spark.read.table("global.silver.olist_customers_dataset")       # clientes
df2  = spark.read.table("global.silver.olist_order_items_dataset")     # itens de pedido
df3  = spark.read.table("global.silver.olist_order_payments_dataset")  # pagamentos
df4  = spark.read.table("global.silver.olist_order_reviews_dataset")   # avaliações
df5  = spark.read.table("global.silver.olist_orders_dataset")          # pedidos
df6  = spark.read.table("global.silver.olist_products_dataset")        # produtos
df7  = spark.read.table("global.silver.olist_sellers_dataset")         # vendedores
df8  = spark.read.table("global.silver.olist_geolocation_dataset")     # geolocalização

# Aliases para clareza (df9 e df10 são duplicados de df6 e df2)
customers   = df1
order_items = df2
payments    = df3
reviews     = df4
orders      = df5
products    = df6
sellers     = df7
geo         = df8

print("Silver carregada. Schemas:\n")
for nome, df in [("customers",df1),("order_items",df2),("payments",df3),
                 ("reviews",df4),("orders",df5),("products",df6),
                 ("sellers",df7),("geo",df8)]:
    print(f"  {nome}: {df.columns}")

# ---------------------------------------------------------------------------
# Helper: gravar como Delta Table
# ---------------------------------------------------------------------------
CATALOG  = "global"
GOLD     = "gold"

def save_gold(df, table_name):
    full = f"{CATALOG}.{GOLD}.{table_name}"
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(full)
    n = df.count()
    print(f" {full}  ({n:,} registos)")

print("\n" + "="*60)


# ===========================================================================
# GOLD 1 — VENDAS E RECEITA POR PERÍODO
# Pergunta: como evoluiu a receita mês a mês?
# ===========================================================================
print("\n GOLD 1 — Vendas e receita por período")

# Pedidos entregues com mês/ano
orders_ok = df5.filter(F.col("order_status") == "delivered") \
    .withColumn("ano",    F.year("order_purchase_timestamp")) \
    .withColumn("mes",    F.month("order_purchase_timestamp")) \
    .withColumn("ano_mes",F.date_format("order_purchase_timestamp","yyyy-MM"))

# Total pago por pedido
pag_por_pedido = df3.groupBy("order_id").agg(
    F.round(F.sum("payment_value"),2).alias("receita_pedido"),
    F.max("payment_installments").alias("parcelas"),
    F.first("payment_type").alias("meio_pagamento")
)

# Itens por pedido
itens_por_pedido = df2.groupBy("order_id").agg(
    F.count("order_item_id").alias("qtd_itens"),
    F.round(F.sum("freight_value"),2).alias("frete_pedido")
)

gold1 = orders_ok \
    .join(pag_por_pedido,   "order_id", "left") \
    .join(itens_por_pedido, "order_id", "left") \
    .groupBy("ano","mes","ano_mes").agg(
        F.count("order_id")                      .alias("total_pedidos"),
        F.countDistinct("customer_id")           .alias("clientes_unicos"),
        F.round(F.sum("receita_pedido"),   2)    .alias("receita_total"),
        F.round(F.avg("receita_pedido"),   2)    .alias("ticket_medio"),
        F.round(F.sum("frete_pedido"),     2)    .alias("frete_total"),
        F.round(F.avg("qtd_itens"),        2)    .alias("media_itens"),
        F.round(F.avg("parcelas"),         2)    .alias("media_parcelas"),
    ) \
    .withColumn("receita_acumulada",
        F.round(F.sum("receita_total").over(
            Window.orderBy("ano","mes")
                  .rowsBetween(Window.unboundedPreceding, 0)), 2)
    ) \
    .withColumn("var_pct_receita",
        F.round(
            (F.col("receita_total") - F.lag("receita_total",1).over(Window.orderBy("ano","mes")))
            / F.lag("receita_total",1).over(Window.orderBy("ano","mes")) * 100, 1)
    ) \
    .orderBy("ano","mes")

gold1.show(10, truncate=False)
save_gold(gold1, "gold_vendas_por_periodo")


# ===========================================================================
# GOLD 2 — PERFORMANCE DE VENDEDORES
# Pergunta: quais vendedores geram mais receita e têm melhor avaliação?
# ===========================================================================
print("\n GOLD 2 — Performance de vendedores")

# Avaliação média por vendedor (items → reviews)
review_seller = df2.join(df4.select("order_id","review_score"), "order_id") \
    .groupBy("seller_id").agg(
        F.round(F.avg("review_score"),2).alias("avg_review"),
        F.count("review_score")        .alias("total_reviews")
    )

gold2 = df2 \
    .join(df5.filter(F.col("order_status")=="delivered")
             .select("order_id","order_purchase_timestamp"), "order_id") \
    .groupBy("seller_id").agg(
        F.count("order_item_id")              .alias("itens_vendidos"),
        F.countDistinct("order_id")           .alias("pedidos_atendidos"),
        F.countDistinct("product_id")         .alias("produtos_distintos"),
        F.round(F.sum("price"),          2)   .alias("receita_total"),
        F.round(F.avg("price"),          2)   .alias("preco_medio"),
        F.round(F.avg("freight_value"),  2)   .alias("frete_medio"),
    ) \
    .join(review_seller, "seller_id", "left") \
    .join(df7.select("seller_id","seller_city","seller_state"), "seller_id","left") \
    .withColumn("ranking_receita",
        F.rank().over(Window.orderBy(F.desc("receita_total")))) \
    .withColumn("tier",
        F.when(F.col("ranking_receita") <=  50, " Top 50")
         .when(F.col("ranking_receita") <= 200, " Top 200")
         .otherwise("Standard")) \
    .orderBy("ranking_receita")

gold2.show(10, truncate=False)
save_gold(gold2, "gold_performance_vendedores")


# ===========================================================================
# GOLD 3 — SEGMENTAÇÃO DE CLIENTES (RFM)
# Pergunta: quais clientes são mais valiosos?
# ===========================================================================
print("\n👥 GOLD 3 — Segmentação de clientes (RFM)")

data_ref = df5.agg(F.max("order_purchase_timestamp")).collect()[0][0]

valor_por_pedido = df3.groupBy("order_id").agg(
    F.round(F.sum("payment_value"),2).alias("valor_pedido")
)

rfm = df5.filter(F.col("order_status")=="delivered") \
    .join(valor_por_pedido, "order_id","left") \
    .groupBy("customer_id").agg(
        F.datediff(F.lit(data_ref),
                   F.max("order_purchase_timestamp"))  .alias("recencia_dias"),
        F.count("order_id")                            .alias("frequencia"),
        F.round(F.sum("valor_pedido"),  2)             .alias("valor_total"),
        F.round(F.avg("valor_pedido"),  2)             .alias("ticket_medio"),
        F.min("order_purchase_timestamp")              .alias("primeira_compra"),
        F.max("order_purchase_timestamp")              .alias("ultima_compra"),
    )

gold3 = rfm \
    .withColumn("r", F.ntile(5).over(Window.orderBy(F.desc("recencia_dias")))) \
    .withColumn("f", F.ntile(5).over(Window.orderBy("frequencia"))) \
    .withColumn("m", F.ntile(5).over(Window.orderBy("valor_total"))) \
    .withColumn("rfm_score", F.col("r")*100 + F.col("f")*10 + F.col("m")) \
    .withColumn("segmento",
        F.when((F.col("r")>=4)&(F.col("f")>=4),             "Campeões")
         .when((F.col("r")>=3)&(F.col("f")>=3),             "Fiéis")
         .when((F.col("r")>=4)&(F.col("f")<=2),             "Novos")
         .when((F.col("r")<=2)&(F.col("f")>=3),             "Em Risco")
         .when((F.col("r")<=2)&(F.col("f")<=2),             "Perdidos")
         .otherwise(                                          "Potencial")) \
    .join(df1.select("customer_id","customer_city","customer_state"),
          "customer_id","left")

gold3.show(10, truncate=False)
save_gold(gold3, "gold_segmentacao_clientes")


# ===========================================================================
# GOLD 4 — SATISFAÇÃO E NPS
# Pergunta: qual o NPS por mês e como a entrega influencia a nota?
# ===========================================================================
print("\n GOLD 4 — Satisfação e NPS")

reviews_enrich = df4 \
    .join(df5.select("order_id","order_purchase_timestamp",
                     "order_delivered_customer_date",
                     "order_estimated_delivery_date"), "order_id","left") \
    .withColumn("dias_entrega",
        F.datediff("order_delivered_customer_date","order_purchase_timestamp")) \
    .withColumn("entregue_no_prazo",
        F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date")) \
    .withColumn("tem_comentario",
        F.col("review_comment_message").isNotNull() &
        (F.trim(F.col("review_comment_message")) != "")) \
    .withColumn("ano_mes",
        F.date_format("order_purchase_timestamp","yyyy-MM"))

gold4 = reviews_enrich.groupBy("ano_mes").agg(
    F.count("review_id")                                        .alias("total_reviews"),
    F.round(F.avg("review_score"),2)                           .alias("avg_score"),
    F.sum(F.when(F.col("review_score")==5,1).otherwise(0))     .alias("notas_5"),
    F.sum(F.when(F.col("review_score")==4,1).otherwise(0))     .alias("notas_4"),
    F.sum(F.when(F.col("review_score")==3,1).otherwise(0))     .alias("notas_3"),
    F.sum(F.when(F.col("review_score")==2,1).otherwise(0))     .alias("notas_2"),
    F.sum(F.when(F.col("review_score")==1,1).otherwise(0))     .alias("notas_1"),
    F.round(F.avg("dias_entrega"),1)                           .alias("avg_dias_entrega"),
    F.sum(F.when(F.col("entregue_no_prazo"),1).otherwise(0))   .alias("entregas_no_prazo"),
    F.sum(F.when(F.col("tem_comentario"),1).otherwise(0))      .alias("com_comentario"),
) \
.withColumn("promotores_pct",
    F.round(F.col("notas_5")/F.col("total_reviews")*100,1)) \
.withColumn("detratores_pct",
    F.round((F.col("notas_1")+F.col("notas_2"))/F.col("total_reviews")*100,1)) \
.withColumn("nps",
    F.round(F.col("promotores_pct")-F.col("detratores_pct"),1)) \
.withColumn("pct_no_prazo",
    F.round(F.col("entregas_no_prazo")/F.col("total_reviews")*100,1)) \
.orderBy("ano_mes")

gold4.show(10, truncate=False)
save_gold(gold4, "gold_satisfacao_nps")


# ===========================================================================
# GOLD 5 — CATEGORIAS DE PRODUTOS
# Pergunta: quais categorias vendem mais e têm melhor avaliação?
# ===========================================================================
print("\n  GOLD 5 — Categorias de produtos")

gold5 = df2 \
    .join(df5.filter(F.col("order_status")=="delivered")
             .select("order_id"), "order_id") \
    .join(df6.select("product_id","product_category_name",
                     "product_weight_g"), "product_id","left") \
    .join(df4.select("order_id","review_score"), "order_id","left") \
    .groupBy("product_category_name").agg(
        F.count("order_item_id")              .alias("itens_vendidos"),
        F.countDistinct("order_id")           .alias("pedidos"),
        F.countDistinct("product_id")         .alias("skus"),
        F.countDistinct("seller_id")          .alias("vendedores"),
        F.round(F.sum("price"),          2)   .alias("receita_total"),
        F.round(F.avg("price"),          2)   .alias("preco_medio"),
        F.round(F.min("price"),          2)   .alias("preco_minimo"),
        F.round(F.max("price"),          2)   .alias("preco_maximo"),
        F.round(F.avg("freight_value"),  2)   .alias("frete_medio"),
        F.round(F.avg("review_score"),   2)   .alias("avg_review"),
        F.round(F.avg("product_weight_g"),0)  .alias("peso_medio_g"),
    ) \
    .withColumn("ranking",
        F.rank().over(Window.orderBy(F.desc("receita_total")))) \
    .withColumn("pct_receita",
        F.round(F.col("receita_total") /
            F.sum("receita_total").over(Window.orderBy(F.lit(1))
                .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))
            * 100, 2)) \
    .orderBy("ranking")

gold5.show(15, truncate=False)
save_gold(gold5, "gold_categorias_produtos")


# ===========================================================================
# GOLD 6 — GEOGRAFIAS
# Pergunta: onde estão os clientes e como varia a entrega por estado?
# ===========================================================================
print("\n  GOLD 6 — Geografias")

valor_estado = df5.filter(F.col("order_status")=="delivered") \
    .join(df1.select("customer_id","customer_state","customer_city"), "customer_id") \
    .join(df3.groupBy("order_id")
             .agg(F.round(F.sum("payment_value"),2).alias("valor")),
          "order_id","left") \
    .withColumn("dias_entrega",
        F.datediff("order_delivered_customer_date","order_purchase_timestamp")) \
    .groupBy("customer_state").agg(
        F.countDistinct("customer_id")   .alias("clientes"),
        F.count("order_id")              .alias("pedidos"),
        F.round(F.sum("valor"),     2)   .alias("receita_total"),
        F.round(F.avg("valor"),     2)   .alias("ticket_medio"),
        F.countDistinct("customer_city") .alias("cidades"),
        F.round(F.avg("dias_entrega"),1) .alias("avg_dias_entrega"),
    )

vendedores_estado = df7.groupBy("seller_state").agg(
    F.countDistinct("seller_id").alias("vendedores")
).withColumnRenamed("seller_state","customer_state")

gold6 = valor_estado \
    .join(vendedores_estado, "customer_state","left") \
    .withColumn("ranking",
        F.rank().over(Window.orderBy(F.desc("receita_total")))) \
    .orderBy("ranking")

gold6.show(27, truncate=False)
save_gold(gold6, "gold_geografias")


# ===========================================================================
# SUMÁRIO FINAL
# ===========================================================================
print(f"""
╔══════════════════════════════════════════════════════╗
║        GOLD OLIST — PIPELINE CONCLUÍDO               ║
╠══════════════════════════════════════════════════════╣
║  ✅  gold_vendas_por_periodo                         ║
║  ✅  gold_performance_vendedores                     ║
║  ✅  gold_segmentacao_clientes  (RFM)                ║
║  ✅  gold_satisfacao_nps        (NPS)                ║
║  ✅  gold_categorias_produtos                        ║
║  ✅  gold_geografias                                 ║
╠══════════════════════════════════════════════════════╣
║  Catálogo: sql_saturday.gold.*                       ║
╚══════════════════════════════════════════════════════╝
""")

spark.sql("SHOW TABLES IN sql_saturday.gold").show(truncate=False)
